In [0]:
dbutils.widgets.text('catalog', 'dev')
catalog = dbutils.widgets.get('catalog')

In [0]:
from pyspark.sql import functions as F

# Documented business-rule window (same as the cleaning pipeline)
DATE_START = '1992-01-01'
DATE_END = '1998-12-31'

# All eight TPC-H source tables
SOURCE_TABLES = ['region', 'nation', 'part', 'supplier', 'customer', 'partsupp', 'orders', 'lineitem']

# ------------------------------------------------------------------
# Results framework: every check records itself and FAILS FAST — the
# first violated critical check immediately persists all recorded
# results and raises a pipeline-stopping error.
# ------------------------------------------------------------------
CHECK_RESULTS = []

def _persist_results_before_fail():
    try:
        results_df = spark.createDataFrame(CHECK_RESULTS)
        results_df.write.mode('overwrite').saveAsTable(f'{catalog}.gold.data_quality_results')
        print(f"DQ results persisted to {catalog}.gold.data_quality_results")
    except Exception as persist_err:
        print(f'WARNING: could not persist DQ results before failing: {persist_err}')

def record_check(check_id, description, passed, details, critical=True):
    CHECK_RESULTS.append({
        'check_id': check_id,
        'description': description,
        'status': 'PASS' if passed else 'FAIL',
        'details': details,
        'critical': critical,
    })
    crit = ' [CRITICAL]' if critical else ''
    print(f"{'PASS' if passed else 'FAIL'}: {description} | {details}{crit}")
    if not passed and critical:
        _persist_results_before_fail()
        raise Exception(
            'DATA QUALITY GATE FAILED — critical check [' + check_id + '] violated: '
            + description + ' (' + details + '). Pipeline stopped.'
        )

def table_exists(name):
    return spark.catalog.tableExists(name)

print(f'Data quality checks will run against catalog: {catalog}')

In [0]:
# Check 1 — All eight source tables were ingested into Bronze

missing, counts = [], {}
for t in SOURCE_TABLES:
    bronze_name = f'{catalog}.bronze.{t}'
    if not table_exists(bronze_name):
        missing.append(bronze_name)
    else:
        n = spark.table(bronze_name).count()
        counts[t] = n
        if n == 0:
            missing.append(f'{bronze_name} (0 rows)')

ingestion_df = spark.createDataFrame(
    [(t, counts.get(t)) for t in SOURCE_TABLES],
    ['source_table', 'bronze_row_count'],
)
display(ingestion_df)

record_check(
    'C1',
    'All eight source tables ingested into Bronze',
    len(missing) == 0,
    f'ingested={len(counts)}/8' + (f'; missing/empty: {missing}' if missing else ''),
)

In [0]:
# Check 2 — Primary keys are populated and unique where required

# Silver PKs (same definitions as the cleaning pipeline) + Gold dimension PKs
PK_SPECS = {
    f'{catalog}.silver.region':      ['r_regionkey'],
    f'{catalog}.silver.nation':      ['n_nationkey'],
    f'{catalog}.silver.part':        ['p_partkey'],
    f'{catalog}.silver.supplier':    ['s_suppkey'],
    f'{catalog}.silver.customer':    ['c_custkey'],
    f'{catalog}.silver.partsupp':    ['ps_partkey', 'ps_suppkey'],
    f'{catalog}.silver.orders':      ['o_orderkey'],
    f'{catalog}.silver.lineitem':    ['l_orderkey', 'l_linenumber'],
    f'{catalog}.gold.dim_customer':  ['customerkey'],
    f'{catalog}.gold.dim_product':   ['productkey'],
    f'{catalog}.gold.dim_supplier':  ['supplierkey'],
    f'{catalog}.gold.dim_geography': ['nationkey'],
}

def check_primary_key(table_name, key_cols):
    df = spark.table(table_name)
    agg = df.agg(
        F.count(F.lit(1)).alias('_total'),
        *[F.sum(F.col(c).isNull().cast('long')).alias(c) for c in key_cols],
    ).collect()[0]
    total = agg['_total']
    null_total = sum(agg[c] or 0 for c in key_cols)
    distinct = df.select(*key_cols).distinct().count()
    duplicate_rows = total - distinct
    passed = total > 0 and null_total == 0 and duplicate_rows == 0
    record_check(
        'C2',
        f"Primary key [{', '.join(key_cols)}] populated & unique in {table_name}",
        passed,
        f'rows={total:,}, null_keys={null_total:,}, duplicate_rows={duplicate_rows:,}',
    )

for tbl, keys in PK_SPECS.items():
    if table_exists(tbl):
        check_primary_key(tbl, keys)
    else:
        record_check('C2', f'Primary key check for {tbl}', False, 'table does not exist')

In [0]:
# Check 3 — Fact table contains no null surrogate keys

fact = spark.table(f'{catalog}.gold.fact_sales')

SURROGATE_KEYS = ['orderkey', 'linenumber', 'customerkey', 'productkey', 'supplierkey', 'nationkey', 'regionkey']

nulls = fact.agg(*[F.sum(F.col(c).isNull().cast('long')).alias(c) for c in SURROGATE_KEYS]).collect()[0].asDict()
null_keys = {k: v for k, v in nulls.items() if v}
fact_rows = fact.count()

record_check(
    'C3',
    'Gold fact_sales has no null surrogate keys',
    not null_keys,
    f"rows={fact_rows:,}; null counts: {null_keys if null_keys else 'none'}",
)

In [0]:
# Check 4 — Numeric measures satisfy the documented business rules
# Rules (as validated in the cleaning pipeline and used by the Gold metric definitions):
#   quantity      > 0
#   extendedprice > 0
#   discount      in [0, 0.10]
#   tax           in [0, 0.08]
#   net_revenue   = extendedprice * (1 - discount)  (Gold metric definition) and > 0
#   orderdate     within [1992-01-01, 1998-12-31]

violations = fact.agg(
    F.sum((F.col('quantity') <= 0).cast('long')).alias('quantity_le_0'),
    F.sum((F.col('extendedprice') <= 0).cast('long')).alias('extendedprice_le_0'),
    F.sum(((F.col('discount') < 0) | (F.col('discount') > 0.10)).cast('long')).alias('discount_out_of_range'),
    F.sum(((F.col('tax') < 0) | (F.col('tax') > 0.08)).cast('long')).alias('tax_out_of_range'),
    F.sum((F.abs(F.col('net_revenue') - F.col('extendedprice') * (1 - F.col('discount'))) > 1e-6).cast('long')).alias('net_revenue_mismatch'),
    F.sum((F.col('net_revenue') <= 0).cast('long')).alias('net_revenue_le_0'),
    F.sum(
        (F.col('orderdate').isNull()
         | (F.col('orderdate') < F.lit(DATE_START).cast('date'))
         | (F.col('orderdate') > F.lit(DATE_END).cast('date'))
        ).cast('long')
    ).alias('orderdate_out_of_range'),
).collect()[0].asDict()

bad = {k: v for k, v in violations.items() if v}
record_check(
    'C4',
    'Numeric measures satisfy documented business rules (fact_sales)',
    not bad,
    f"violations: {bad if bad else 'none'}",
)

In [0]:
# Check 5 — Gold revenue and row counts reconcile with Silver

silver_lineitem = spark.table(f'{catalog}.silver.lineitem')
silver_orders = spark.table(f'{catalog}.silver.orders')

silver_metrics = silver_lineitem.agg(
    F.count(F.lit(1)).alias('row_count'),
    F.sum(F.col('l_extendedprice') * (1 - F.col('l_discount'))).alias('net_revenue'),
    F.countDistinct('l_orderkey').alias('distinct_orders'),
).collect()[0]

gold_metrics = fact.agg(
    F.count(F.lit(1)).alias('row_count'),
    F.sum('net_revenue').alias('net_revenue'),
    F.countDistinct('orderkey').alias('distinct_orders'),
).collect()[0]

# Revenue compared with a tiny relative tolerance for floating-point rounding (decimals -> float)
rev_silver = float(silver_metrics['net_revenue'])
rev_gold = float(gold_metrics['net_revenue'])
rev_diff = abs(rev_silver - rev_gold)
rev_ok = rev_diff <= abs(rev_silver) * 1e-9

recon_df = spark.createDataFrame([
    ('fact row count', float(silver_metrics['row_count']), float(gold_metrics['row_count']), silver_metrics['row_count'] == gold_metrics['row_count']),
    ('net revenue', rev_silver, rev_gold, rev_ok),
    ('distinct orders', float(silver_metrics['distinct_orders']), float(gold_metrics['distinct_orders']), silver_metrics['distinct_orders'] == gold_metrics['distinct_orders']),
], ['metric', 'silver_value', 'gold_value', 'match'])
display(recon_df)

record_check(
    'C5',
    'Gold fact_sales reconciles with Silver (row count, net revenue, distinct orders)',
    bool(recon_df.filter('match = false').count() == 0),
    f"silver_rows={silver_metrics['row_count']:,}, gold_rows={gold_metrics['row_count']:,}, revenue_diff={rev_diff}, silver_orders={silver_metrics['distinct_orders']:,}, gold_orders={gold_metrics['distinct_orders']:,}",
)

In [0]:
# Check 6 — Rejected-record counts are reported (informational, not pipeline-stopping)

quarantine_rows = []
for t in SOURCE_TABLES:
    q = f'{catalog}.silver.{t}_quarantine'
    if table_exists(q):
        n = spark.table(q).count()
    else:
        n = None  # no quarantine table = no rejections were produced for this table
    quarantine_rows.append((t, n))

rejected_df = spark.createDataFrame(quarantine_rows, ['source_table', 'rejected_row_count'])
display(rejected_df)

total_rejected = sum(n for _, n in quarantine_rows if n is not None)
record_check(
    'C6',
    'Rejected-record counts reported (Silver quarantine tables)',
    True,
    f'total_rejected={total_rejected:,}',
    critical=False,
)

In [0]:
# Summary — persist results and raise a pipeline-stopping error on any critical failure

results_df = spark.createDataFrame(CHECK_RESULTS)
results_df = results_df.orderBy('check_id')
display(results_df)

results_df.write.mode('overwrite').saveAsTable(f'{catalog}.gold.data_quality_results')
print(f"DQ results persisted to {catalog}.gold.data_quality_results")

critical_failures = [r for r in CHECK_RESULTS if r['critical'] and r['status'] == 'FAIL']

if critical_failures:
    failure_summary = '; '.join("[" + r['check_id'] + "] " + r['description'] + " (" + r['details'] + ")" for r in critical_failures)
    raise Exception(
        'DATA QUALITY GATE FAILED — ' + str(len(critical_failures)) + ' critical check(s) violated. '
        'Pipeline stopped. Failures: ' + failure_summary
    )

print('ALL CRITICAL DATA QUALITY CHECKS PASSED (' + str(len(CHECK_RESULTS)) + ' checks recorded, ' + format(total_rejected, ',') + ' rejected records reported).')